### Lab: Unsupervised and Supervised ML for Water Extent Monitoring

This week, we'll apply k-means clustering and a simple linear classifier to Landsat 8 imagery to monitor changes in a lake in Nevada over time. As usual, you'll generate your own code, working mostly from this [spectral clustering tutorial from the Landsat ML Cookbook](https://projectpythia.org/landsat-ml-cookbook/notebooks/spectral-clustering-pc/).

A couple of small notes here:
- For ease of visualization, we recommend using static `matplotlib` plots rather than interactive `hvplot` ones.
- Instead of spectral clustering, we'll use k-means clustering from `dask-ml`, since there seems to be some kind of bug in the spectral clustering in the latest version of Dask.

In [19]:
# SETUP:

import numpy as np
import odc.stac
import pandas as pd
import planetary_computer
import pystac_client
import xarray as xr
from dask.distributed import Client
from dask_ml.cluster import SpectralClustering
from pystac.extensions.eo import EOExtension as eo
import pyproj

# Viz
import hvplot.xarray
from IPython.display import Image

ModuleNotFoundError: No module named 'dask_ml'

### Part 1: Data Preparation

1. Per the Landsat ML Cookbook tutorial, query Landsat 8 data from Planetary Computer using the following parameters: 

```
bbox = [-118.89, 38.54, -118.57, 38.84]  # Region over a lake in Nevada, USA
datetime = "2017-06-01/2017-09-30"  # Summer months of 2017
collection = "landsat-c2-l2"
platform = "landsat-8"
cloudy_less_than = 1  # percent
```
Examine the metadata (e.g., bands) and visualize the pre-rendered overview of the scene.

2. Load the data with `odc-stac`. What does `da_2017` look like when you view it? (Just run `da_2017` in a cell to see the output.) What dimensions does it have? What size is it, and how does its chunk size relate to the total size?

3. Flatten your array so that it is two-dimensional: `n_samples` by `n_features`. View this the same way you did before--what does it look like now?

4. Standardize your data following the Landsat ML Cookbook tutorial. Why is standardization important for k-means clustering? (Hint: Euclidian distance.)

In [4]:
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)

bbox = [-118.89, 38.54, -118.57, 38.84]  # Region over a lake in Nevada, USA
datetime = "2017-06-01/2017-09-30"  # Summer months of 2017
collection = "landsat-c2-l2"
platform = "landsat-8"
cloudy_less_than = 1  # percent

search = catalog.search(
    collections=["landsat-c2-l2"],
    bbox=bbox,
    datetime=datetime,
    query={"eo:cloud_cover": {"lt": cloudy_less_than}, "platform": {"in": [platform]}},
)
items = search.item_collection()
print(f"Returned {len(items)} Items:")
[[i, item.id] for i, item in enumerate(items)]

Returned 3 Items:


[[0, 'LC08_L2SP_042033_20170718_02_T1'],
 [1, 'LC08_L2SP_042033_20170702_02_T1'],
 [2, 'LC08_L2SP_042033_20170616_02_T1']]

In [11]:
item = items[0]
item

<Item id=LC08_L2SP_042033_20170718_02_T1>

In [12]:
Image(url=items[0].assets["rendered_preview"].href)

In [16]:
assets = []
for _, asset in item.assets.items():
    try:
        assets.append(asset.extra_fields["eo:bands"][0])
    except:
        pass

cols_ordered = [
    "common_name",
    "description",
    "name",
    "center_wavelength",
    "full_width_half_max",
]
bands = pd.DataFrame.from_dict(assets)[cols_ordered]
bands

ds_2017 = odc.stac.stac_load(
    [item],
    bands=bands.common_name.values,
    bbox=bbox,
    chunks={},  # <-- use Dask
).isel(time=0)

epsg = item.properties["proj:code"]
ds_2017.attrs["crs"] = f"epsg:{epsg}"

da_2017 = ds_2017.to_array(dim="band")
da_2017


<xarray.DataArray (band: 8, y: 1128, x: 950)> Size: 17MB
dask.array<stack, shape=(8, 1128, 950), dtype=uint16, chunksize=(1, 1128, 950), chunktype=numpy.ndarray>
Coordinates:
  * band         (band) object 64B 'red' 'blue' 'green' ... 'swir22' 'coastal'
  * y            (y) float64 9kB 4.301e+06 4.301e+06 ... 4.267e+06 4.267e+06
  * x            (x) float64 8kB 3.353e+05 3.353e+05 ... 3.637e+05 3.638e+05
    spatial_ref  int32 4B 32611
    time         datetime64[ns] 8B 2017-07-18T18:33:10.564878
Attributes:
    crs:      epsg:EPSG:32611

In [17]:
flattened_xda = da_2017.stack(z=("x", "y"))  # flatten each band
flattened_t_xda = flattened_xda.transpose("z", "band")
flattened_t_xda

<xarray.DataArray (z: 1071600, band: 8)> Size: 17MB
dask.array<transpose, shape=(1071600, 8), dtype=uint16, chunksize=(1071600, 1), chunktype=numpy.ndarray>
Coordinates:
  * z            (z) object 9MB MultiIndex
  * band         (band) object 64B 'red' 'blue' 'green' ... 'swir22' 'coastal'
    spatial_ref  int32 4B 32611
    time         datetime64[ns] 8B 2017-07-18T18:33:10.564878
  * x            (z) float64 9MB 3.353e+05 3.353e+05 ... 3.638e+05 3.638e+05
  * y            (z) float64 9MB 4.301e+06 4.301e+06 ... 4.267e+06 4.267e+06
Attributes:
    crs:      epsg:EPSG:32611

In [18]:
# Standardization

with xr.set_options(keep_attrs=True):
    rescaled_xda = (flattened_t_xda - flattened_t_xda.mean()) / flattened_t_xda.std()
rescaled_xda

<xarray.DataArray (z: 1071600, band: 8)> Size: 69MB
dask.array<truediv, shape=(1071600, 8), dtype=float64, chunksize=(1071600, 1), chunktype=numpy.ndarray>
Coordinates:
  * z            (z) object 9MB MultiIndex
  * band         (band) object 64B 'red' 'blue' 'green' ... 'swir22' 'coastal'
    spatial_ref  int32 4B 32611
    time         datetime64[ns] 8B 2017-07-18T18:33:10.564878
  * x            (z) float64 9MB 3.353e+05 3.353e+05 ... 3.638e+05 3.638e+05
  * y            (z) float64 9MB 4.301e+06 4.301e+06 ... 4.267e+06 4.267e+06
Attributes:
    crs:      epsg:EPSG:32611

### Part 2: K-Means Clustering

5. Initialize your Dask cluster and use `dask_ml` to fit a KMeans model to the data. The tutorial uses 4 clusters, which is fine for our purposes. What bands are we loading into our clusters? 

6. Map the resulting clusters next to the original image (you can just use a grayscale band for this). How do the clusters correspond to land versus water? Do you notice any inconsistencies or noise?

In [ ]:
client = Client(processes=False)
client

X = client.persist(rescaled_xda)
X.shape

clf = SpectralClustering(
    n_clusters=4,
    random_state=0,
    gamma=None,
    kmeans_params={"init_max_iter": 5},
    persist_embedding=True,
)